# Day 08：面向对象编程（OOP Part 1）—— 用 `ChatMessage` 类封装大模型对话消息

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 8 天！我是你的主讲兼 Tech Lead 张工。在 Day 05 我们用裸字典 `{"role": "user", "content": "..."}` 跑通了 JSON 解析与存储，但在真实的企业工程里，裸字典就像“没有安检的散装包裹”——拼错一个键名、塞入非法角色都不会立刻报错。今天我们正式进入 **OOP（面向对象编程）**：手写智枢平台的第一个核心实体类 `ChatMessage`，把**数据属性**与**校验、序列化、Token 估算行为**封装到一起！

## 1. 第一步：为什么需要类（`class`）？—— 对比裸字典与 OOP 的差异
先来看一个工程中常见痛点：使用裸字典时，即使把 `"user"` 拼成 `"usr"`，Python 也会悄无声息地放行，直到把请求发给 DeepSeek/OpenAI API 才报 400 错误。
而通过 `class ChatMessage` 的构造函数 `__init__`，我们可以在对象诞生的一瞬间完成校验（**Fail-Fast 快速失败原则**）。

In [1]:
# 1. 裸字典的隐患：拼写错误不会被发现
bad_dict = {"role": "usr", "content": "你好"}
print("裸字典不会报错:", bad_dict)

# 2. 定义允许的角色白名单（类属性，所有实例共享）
VALID_ROLES = ("system", "user", "assistant", "tool")

# 3. 演示在构造阶段进行校验的思路
def check_role(role: str) -> None:
    if role not in VALID_ROLES:
        raise ValueError(f"无效的消息角色: {role!r}，仅支持 {VALID_ROLES}")

try:
    check_role("usr")
except ValueError as err:
    print("提前拦截非法角色:", err)

裸字典不会报错: {'role': 'usr', 'content': '你好'}
提前拦截非法角色: 无效的消息角色: 'usr'，仅支持 ('system', 'user', 'assistant', 'tool')


## 2. 第二步：定义完整的 `ChatMessage` 类（`__init__`、实例方法、`@classmethod`、`@staticmethod`）
在面向对象设计中，我们需要分清三种方法的使用场景：
1. **实例方法（`self`）**：如 `to_dict(self)` 和 `estimate_tokens(self)`，需要读取具体某条消息的 `self.role`、`self.content`、`self.name`。
2. **类方法（`@classmethod` + `cls`）**：如 `from_dict(cls, data)`，用于从外部字典**构造并返回一个新实例**（工厂模式）。
3. **静态方法（`@staticmethod`）**：如 `is_valid_role(role)`，纯工具校验逻辑，**既不需要 `self` 也不需要 `cls`**，不实例化也能直接调用。

In [2]:
from typing import Any


class ChatMessage:
    """大模型标准对话消息实体类。"""

    # 类属性：定义合法的消息角色白名单
    VALID_ROLES: tuple[str, ...] = ("system", "user", "assistant", "tool")

    def __init__(self, role: str, content: str, name: str | None = None) -> None:
        # 1. 调用静态方法校验 role 是否合法（Fail-Fast）
        if not self.is_valid_role(role):
            allowed = ", ".join(self.VALID_ROLES)
            raise ValueError(f"无效的消息角色: {role!r}，合法角色仅限: {allowed}")

        # 2. 校验 content 与可选字段 name 的数据类型
        if not isinstance(content, str):
            raise TypeError(f"content 必须为 str，实际收到: {type(content).__name__}")
        if name is not None and not isinstance(name, str):
            raise TypeError(f"name 必须为 str 或 None，实际收到: {type(name).__name__}")

        # 3. 将校验通过的数据绑定到实例属性 self 上
        self.role: str = role
        self.content: str = content
        self.name: str | None = name

    def to_dict(self) -> dict[str, Any]:
        """实例方法：序列化为 OpenAI / DeepSeek API 兼容字典。"""
        payload: dict[str, Any] = {
            "role": self.role,
            "content": self.content,
        }
        # 仅当 name 不为 None 时才写入字典，避免发送多余的 null 字段
        if self.name is not None:
            payload["name"] = self.name
        return payload

    def estimate_tokens(self) -> int:
        """实例方法：启发式估算当前消息的 Token 消耗量。"""
        if not self.content:
            return 0
        # 统计中文汉字（Unicode 范围 \u4e00 ~ \u9fff）数量
        cjk_count = sum(1 for ch in self.content if "\u4e00" <= ch <= "\u9fff")
        # 其余字符（英文、数字、标点、空格）数量
        other_count = len(self.content) - cjk_count
        # 中文约 1.5 tokens/字，英文标点约 0.3 tokens/字符，非空至少返回 1
        return max(1, round(cjk_count * 1.5 + other_count * 0.3))

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "ChatMessage":
        """类方法（工厂方法）：从字典反序列化构造 ChatMessage 实例。"""
        if not isinstance(data, dict):
            raise ValueError("from_dict 参数必须为 dict 类型")
        if "role" not in data or "content" not in data:
            raise ValueError("消息字典必须包含 'role' 和 'content' 字段")
        # 使用 cls(...) 构造实例，支持未来子类继承扩展
        return cls(
            role=data["role"],
            content=data["content"],
            name=data.get("name"),
        )

    @staticmethod
    def is_valid_role(role: str) -> bool:
        """静态方法：无需实例化即可判断角色字符串是否合法。"""
        return isinstance(role, str) and role in ChatMessage.VALID_ROLES

    def __repr__(self) -> str:
        name_part = f", name={self.name!r}" if self.name is not None else ""
        return f"ChatMessage(role={self.role!r}, content={self.content!r}{name_part})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, ChatMessage):
            return NotImplemented
        return (
            self.role == other.role
            and self.content == other.content
            and self.name == other.name
        )


print("ChatMessage 类定义成功！合法角色:", ChatMessage.VALID_ROLES)

ChatMessage 类定义成功！合法角色: ('system', 'user', 'assistant', 'tool')


## 3. 第三步：实例化对象、调用实例方法与异常防御测试
下面我们创建几条真实业务消息（包含 `system`、`user`、`tool`、`assistant`），观察 `to_dict()`、`estimate_tokens()` 的输出，并验证非法角色会被构造函数精准拦截。

In [3]:
# 1. 创建多条不同角色的 ChatMessage 实例
sys_msg = ChatMessage(role="system", content="你是星辰数科智枢平台企业级 AI 助手。")
usr_msg = ChatMessage(role="user", content="请帮我查询上海今天的天气。", name="chenmo")
tool_msg = ChatMessage(role="tool", content='{"city": "上海", "temp": 22}', name="query_weather")
ast_msg = ChatMessage(role="assistant", content="上海今天气温 22°C，天气晴朗。")

# 2. 打印调试表示 (__repr__)、序列化字典与 Token 估算
for m in (sys_msg, usr_msg, tool_msg, ast_msg):
    print("对象:", repr(m))
    print("字典:", m.to_dict(), "| 预估 Token:", m.estimate_tokens())
    print("-" * 50)

# 3. 验证非法角色会被 __init__ 拦截
try:
    ChatMessage(role="superadmin", content="越权测试")
except ValueError as exc:
    print("成功拦截非法角色:", exc)

对象: ChatMessage(role='system', content='你是星辰数科智枢平台企业级 AI 助手。')
字典: {'role': 'system', 'content': '你是星辰数科智枢平台企业级 AI 助手。'} | 预估 Token: 24
--------------------------------------------------
对象: ChatMessage(role='user', content='请帮我查询上海今天的天气。', name='chenmo')
字典: {'role': 'user', 'content': '请帮我查询上海今天的天气。', 'name': 'chenmo'} | 预估 Token: 18
--------------------------------------------------
对象: ChatMessage(role='tool', content='{"city": "上海", "temp": 22}', name='query_weather')
字典: {'role': 'tool', 'content': '{"city": "上海", "temp": 22}', 'name': 'query_weather'} | 预估 Token: 10
--------------------------------------------------
对象: ChatMessage(role='assistant', content='上海今天气温 22°C，天气晴朗。')
字典: {'role': 'assistant', 'content': '上海今天气温 22°C，天气晴朗。'} | 预估 Token: 17
--------------------------------------------------
成功拦截非法角色: 无效的消息角色: 'superadmin'，合法角色仅限: system, user, assistant, tool


## 4. 第四步：静态方法 `is_valid_role` 与类方法 `from_dict` 实战
在接入外部 HTTP 请求或读取本地历史会话 JSON 时，我们经常要先用 `ChatMessage.is_valid_role()` 做前置校验，再用 `ChatMessage.from_dict()` 批量把 JSON 字典列表还原成强类型的 `ChatMessage` 对象列表。

In [4]:
import json
import tempfile
from pathlib import Path

# 1. 无需创建对象，直接通过类名调用 @staticmethod
for role_name in ("user", "tool", "guest", 123):
    print(f"is_valid_role({role_name!r}) -> {ChatMessage.is_valid_role(role_name)}")

# 2. 模拟将 ChatMessage 列表序列化为中文 JSON 文件，再读取恢复为对象列表
history = [sys_msg, usr_msg, ast_msg]

with tempfile.TemporaryDirectory() as tmp_dir:
    file_path = Path(tmp_dir) / "session_day08.json"
    # 写入阶段：对象 -> to_dict() -> json.dump(ensure_ascii=False)
    serialized = [msg.to_dict() for msg in history]
    file_path.write_text(json.dumps(serialized, ensure_ascii=False, indent=2), encoding="utf-8")

    # 读取阶段：json.loads() -> from_dict() -> 对象列表
    raw_list = json.loads(file_path.read_text(encoding="utf-8"))
    restored = [ChatMessage.from_dict(item) for item in raw_list]

    print("\n恢复后的对象列表:", restored)
    # 得益于 __eq__ 魔术方法，可以直接比较对象列表是否完全一致
    assert restored == history
    print("OOP 往返持久化校验 100% 一致！")

is_valid_role('user') -> True
is_valid_role('tool') -> True
is_valid_role('guest') -> False
is_valid_role(123) -> False

恢复后的对象列表: [ChatMessage(role='system', content='你是星辰数科智枢平台企业级 AI 助手。'), ChatMessage(role='user', content='请帮我查询上海今天的天气。', name='chenmo'), ChatMessage(role='assistant', content='上海今天气温 22°C，天气晴朗。')]
OOP 往返持久化校验 100% 一致！


## 5. 课堂小结与明日预告
- **今日收获**：我们用 `ChatMessage` 完成了从“散装裸字典”到“面向对象实体类”的跃迁，掌握了 `__init__` 构造校验、实例方法（`self`）、`@classmethod` 工厂方法（`cls`）与 `@staticmethod` 静态校验器。
- **明日预告（Day 09）**：我们将学习 OOP 进阶（封装、继承与组合），构建 `Conversation` 对话管理器类，实现多轮上下文窗口裁剪与智能体状态管理！